# Qwen 2.5 7B 띄어쓰기 교정 모델 훈련

조선어 규범집 기반 띄어쓰기 교정 모델 파인튜닝

In [ ]:
# 1. 라이브러리 설치
!pip install -q torch transformers peft accelerate bitsandbytes datasets

In [ ]:
# 2. GPU 확인
import torch
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# 3. Google Drive 마운트 (선택사항)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. 훈련 데이터 다운로드
# GitHub에서 데이터 다운로드 또는 직접 업로드
import json
import os

# 데이터 디렉토리 생성
os.makedirs('/content/data', exist_ok=True)

# 여기에 데이터 파일을 업로드하거나 URL에서 다운로드
# 예: !wget -O /content/data/spacing_correction_train.jsonl <URL>

In [ ]:
# 5. 모델 및 토크나이저 로드
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# 4-bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("모델 로딩 중...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("모델 로딩 완료!")

In [ ]:
# 6. LoRA 설정
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 7. 데이터 준비
from datasets import Dataset

MAX_LENGTH = 512

def format_prompt(item):
    instruction = item["instruction"]
    input_text = item.get("input", "")
    
    if input_text:
        prompt = f"""<|im_start|>system
당신은 조선어(한국어) 띄어쓰기 교정 전문가입니다.<|im_end|>
<|im_start|>user
{instruction}
{input_text}<|im_end|>
<|im_start|>assistant
{item["output"]}<|im_end|>"""
    else:
        prompt = f"""<|im_start|>system
당신은 조선어(한국어) 띄어쓰기 교정 전문가입니다.<|im_end|>
<|im_start|>user
{instruction}<|im_end|>
<|im_start|>assistant
{item["output"]}<|im_end|>"""
    return prompt

def load_data(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

# 데이터 로드 (파일 경로 수정 필요)
train_data = load_data('/content/data/spacing_correction_train.jsonl')
valid_data = load_data('/content/data/spacing_correction_valid.jsonl')

print(f"훈련 데이터: {len(train_data)}개")
print(f"검증 데이터: {len(valid_data)}개")

In [ ]:
# 8. 토크나이징
def tokenize(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )

train_prompts = [format_prompt(item) for item in train_data]
valid_prompts = [format_prompt(item) for item in valid_data]

train_dataset = Dataset.from_dict({"text": train_prompts})
valid_dataset = Dataset.from_dict({"text": valid_prompts})

train_dataset = train_dataset.map(tokenize, batched=True, remove_columns=["text"])
valid_dataset = valid_dataset.map(tokenize, batched=True, remove_columns=["text"])

print(f"토크나이징 완료!")

In [ ]:
# 9. 훈련
from transformers import TrainingArguments, Trainer

OUTPUT_DIR = "/content/qwen_spacing_model"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    logging_steps=10,
    eval_steps=100,
    save_steps=500,
    learning_rate=2e-4,
    fp16=True,
    optim="paged_adamw_8bit",
    save_total_limit=3,
    load_best_model_at_end=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    tokenizer=tokenizer,
)

print("훈련 시작!")
trainer.train()

In [ ]:
# 10. 모델 저장
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"모델 저장 완료: {OUTPUT_DIR}")

In [ ]:
# 11. 테스트
def test_model(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.1)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_prompts = [
    "다음 문장의 띄어쓰기 오류를 수정하라.\n학교가다",
    "다음 문장의 띄어쓰기 오류를 수정하라.\n중 화 인 민 공 화 국",
    "다음 문장의 띄어쓰기 오류를 수정하라.\n한그루의나무",
]

for prompt in test_prompts:
    print(f"입력: {prompt}")
    print(f"출력: {test_model(prompt)}")
    print("-" * 50)